# TravelPlanner LangSmith Trace Error Analysis

This notebook analyzes LangSmith trace exports for the multi-agent TravelPlanner system described in `paper.tex`. It is intentionally built around `example_export/manifest.jsonl`, so adding more exported traces should only require appending rows to the manifest and corresponding run files.

Focus areas:

- repeated tool calls and duplicated exported observations
- repeated or near-duplicate search queries
- dead-loop candidates in the execution loop
- timeouts, incomplete runs, and explicit errors
- cascading errors across subagents, tools, plan mutation, and review

The trace export contains a root artifact JSON per query plus flat LangSmith run and extracted tool-call tables. The notebook distinguishes raw exported rows from de-duplicated tool calls because LangSmith flat exports can observe the same tool call multiple times at different run depths.

## Paper Context

The system is an orchestrator-worker LangGraph pipeline. The constraint agent extracts normalized requirements; the planner produces a non-binding task list; the execution agent runs a ReAct-style loop over domain subagent tools and TravelPlan mutation tools; the quality reviewer validates the assembled plan and can trigger repair.

The paper's tool-use taxonomy motivates the notebook metrics:

- T2 missing mandatory parameters: search/tool query lacks dates, locations, or other required constraints
- T3 redundant or infinite loops: same query or same tool action is reissued without meaningful parameter change
- T4 misinterpreting outputs: later plan slots or review feedback contradicts tool/evidence text
- T5 link rot or empty crawl: evidence links are brittle or unavailable
- T6 cross-agent data corruption: bad or mismatched data propagates from subagent output into plan slots

In [ ]:
from __future__ import annotations

import json
import math
import re
from collections import Counter, defaultdict
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import unquote_plus, urlparse

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
from IPython.display import Markdown, display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

sns.set_theme(style='whitegrid', context='talk')
pd.set_option('display.max_columns', 120)
pd.set_option('display.max_colwidth', 220)

ROOT = Path.cwd()
EXPORT_DIR = ROOT / 'example_export'
MANIFEST_PATH = EXPORT_DIR / 'manifest.jsonl'
RUNS_CSV_PATH = EXPORT_DIR / 'langsmith_runs.csv'
TOOL_CALLS_PATH = EXPORT_DIR / 'tool_calls.jsonl'

ERROR_RE = re.compile(r'timeout|timed out|exception|traceback|failed|failure|error|rate limit|api failed|tool failed|could not|unable to|forbidden|403|empty|missing_info|missing info', re.I)
TIMEOUT_RE = re.compile(r'timeout|timed out|deadline|cancelled|canceled|max.*iterations|max.*steps|recursion', re.I)
MISSING_PARAM_RE = re.compile(r'\b(restaurant|hotel|flight|attraction|museum|route|transit|train|ferry|ticket|opening|hours|meal|lunch|dinner|breakfast)\b', re.I)
DATE_RE = re.compile(r'\b(20\d{2}|jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec|january|february|march|april|june|july|august|september|october|november|monday|tuesday|wednesday|thursday|friday|saturday|sunday|day \d+)\b', re.I)
LOCATION_RE = re.compile(r'\b(from|to|in|near|at|rome|italy|airport|station|hotel|city|downtown|center|centre)\b', re.I)
PLAN_MUTATION_TOOLS = {'init_plan', 'add_day', 'add_slot', 'insert_slot', 'delete_slot', 'remove_day', 'write_todos'}
SEARCH_TOOLS = {'search_web', 'search_restaurants', 'search_attractions', 'search_hotels', 'search_flights', 'build_place_distance_graph', 'check_route_timing', 'distance_between_places', 'closest_places_to_target'}

def read_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    rows = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def parse_json_maybe(value):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None
    if isinstance(value, (dict, list)):
        return value
    text = str(value)
    if not text.strip():
        return None
    try:
        return json.loads(text)
    except Exception:
        return None

def canonical_text(value) -> str:
    if value is None:
        return ''
    if not isinstance(value, str):
        value = json.dumps(value, ensure_ascii=False, sort_keys=True)
    value = unquote_plus(value.lower())
    value = re.sub(r'https?://\S+', ' ', value)
    value = re.sub(r'[^a-z0-9\s.-]', ' ', value)
    return re.sub(r'\s+', ' ', value).strip()

def resolve_trace_file(local_file: str) -> Path:
    candidates = [ROOT / local_file, EXPORT_DIR / local_file, EXPORT_DIR.parent / local_file]
    for path in candidates:
        if path.exists():
            return path
    return candidates[0]

def parse_dt(value):
    if pd.isna(value) or not str(value).strip():
        return pd.NaT
    return pd.to_datetime(value, errors='coerce', utc=True)


## Quick JSON Recon With jq

These shell checks are intentionally included because `jq` is the fastest way to sanity-check new export files before the Python analysis loads them.

In [ ]:
%%bash
set -euo pipefail
echo 'Manifest fields:'
jq -r 'keys_unsorted' example_export/manifest.jsonl
echo
echo 'Manifest summary:'
jq -s '{traces:length, projects:([.[].project_name] | unique), validation_passed:([.[].validation_passed] | group_by(.) | map({value:.[0], count:length}))}' example_export/manifest.jsonl
echo
echo 'Raw vs unique tool-call observations:'
jq -s '{raw_rows:length, unique_tool_call_ids:([.[].tool_call_id] | unique | length), tools:([.[].tool_name] | group_by(.) | map({tool:.[0], rows:length}) | sort_by(-.rows))}' example_export/tool_calls.jsonl

## Load Manifest, Root Artifacts, Flat Runs, And Tool Calls

In [ ]:
manifest = pd.DataFrame(read_jsonl(MANIFEST_PATH))
if manifest.empty:
    raise FileNotFoundError(f'No manifest rows found at {MANIFEST_PATH}')

manifest['trace_file'] = manifest['local_file'].map(lambda p: str(resolve_trace_file(p)))
manifest['trace_file_exists'] = manifest['trace_file'].map(lambda p: Path(p).exists())
manifest['query_short'] = manifest['query'].fillna('').str.slice(0, 120)

root_artifacts = {}
for row in manifest.itertuples(index=False):
    path = Path(row.trace_file)
    if path.exists():
        with path.open('r', encoding='utf-8') as f:
            root_artifacts[row.root_run_id] = json.load(f)

runs = pd.read_csv(RUNS_CSV_PATH) if RUNS_CSV_PATH.exists() else pd.DataFrame()
if not runs.empty:
    for col in ['start_time', 'end_time']:
        runs[col + '_dt'] = runs[col].map(parse_dt)
    runs['duration_s'] = (runs['end_time_dt'] - runs['start_time_dt']).dt.total_seconds()
    for col in ['inputs_json', 'outputs_json', 'metadata_json', 'tags_json']:
        if col in runs:
            runs[col.replace('_json', '')] = runs[col].map(parse_json_maybe)
    runs['has_error'] = runs['error'].fillna('').astype(str).str.contains(ERROR_RE)
    runs['timeout_flag'] = runs[['error', 'name', 'inputs_json', 'outputs_json']].fillna('').astype(str).agg(' '.join, axis=1).str.contains(TIMEOUT_RE)
    runs['root_run_id'] = runs.get('local_root_run_id', pd.Series(index=runs.index, dtype=object)).fillna(runs.get('trace_id'))

tool_raw = pd.DataFrame(read_jsonl(TOOL_CALLS_PATH))
if not tool_raw.empty:
    tool_raw['args'] = tool_raw['tool_args_json'].map(parse_json_maybe)
    tool_raw['query'] = tool_raw['args'].map(lambda x: (x or {}).get('query') or (x or {}).get('url') or (x or {}).get('title') or json.dumps(x or {}, ensure_ascii=False, sort_keys=True))
    tool_raw['query_norm'] = tool_raw['query'].map(canonical_text)
    tool_raw['args_norm'] = tool_raw['args'].map(lambda x: canonical_text(x or {}))
    tool_raw['is_search_tool'] = tool_raw['tool_name'].isin(SEARCH_TOOLS)
    tool_raw['is_plan_mutation'] = tool_raw['tool_name'].isin(PLAN_MUTATION_TOOLS)
    tool_raw['missing_date_hint'] = tool_raw['query'].fillna('').str.contains(MISSING_PARAM_RE) & ~tool_raw['query'].fillna('').str.contains(DATE_RE)
    tool_raw['missing_location_hint'] = tool_raw['query'].fillna('').str.contains(MISSING_PARAM_RE) & ~tool_raw['query'].fillna('').str.contains(LOCATION_RE)
    dedupe_cols = ['thread_id', 'tool_call_id', 'tool_name', 'args_norm']
    tool_calls = tool_raw.drop_duplicates(dedupe_cols).copy().reset_index(drop=True)
    tool_calls['seq'] = tool_calls.groupby('thread_id').cumcount()
else:
    tool_calls = pd.DataFrame()

display(Markdown(f'Loaded **{len(manifest)}** manifest traces, **{len(root_artifacts)}** root artifacts, **{len(runs)}** flat LangSmith runs, **{len(tool_raw)}** raw tool observations, and **{len(tool_calls)}** de-duplicated tool calls.'))
display(manifest[['root_run_id', 'thread_id', 'project_name', 'validation_passed', 'validation_attempts', 'days', 'trace_file_exists', 'query_short']])

## Root Artifact Summary

Root artifacts contain the final typed TravelPlanner state: normalized constraints, planner tasks, TravelPlan days and slots, todo state, validation result, and per-agent histories when present.

In [ ]:
summary_rows, task_rows, slot_rows, todo_rows, history_rows = [], [], [], [], []

for root_id, artifact in root_artifacts.items():
    outputs = artifact.get('outputs') or {}
    plan = outputs.get('travelplan') or {}
    days = plan.get('days') or []
    slots = [slot for day in days for slot in (day.get('slots') or [])]
    costs = [slot.get('cost') for slot in slots if isinstance(slot.get('cost'), (int, float))]
    links = [link for slot in slots for link in (slot.get('links') or []) if isinstance(link, str)]
    summary_rows.append({
        'root_run_id': root_id,
        'query': outputs.get('query'),
        'validation_passed': outputs.get('validation_passed'),
        'validation_attempts': outputs.get('validation_attempts'),
        'validation_feedback': outputs.get('validation_feedback'),
        'constraint_count': len(outputs.get('normalized_constraints') or {}),
        'task_count': len(outputs.get('task_list') or []),
        'day_count': len(days),
        'slot_count': len(slots),
        'total_cost': float(np.nansum(costs)) if costs else 0.0,
        'link_count': len(links),
        'unique_link_domains': len({urlparse(link).netloc.lower().replace('www.', '') for link in links}),
    })
    for i, task in enumerate(outputs.get('task_list') or []):
        task_rows.append({'root_run_id': root_id, 'task_index': i, **task})
    for i, todo in enumerate(outputs.get('todos') or []):
        todo_rows.append({'root_run_id': root_id, 'todo_index': i, **todo})
    for day in days:
        for i, slot in enumerate(day.get('slots') or []):
            links = slot.get('links') or []
            text = ' '.join(str(slot.get(k, '')) for k in ['name', 'description', 'location', 'notes'])
            slot_rows.append({
                'root_run_id': root_id,
                'day_index': day.get('index'),
                'date': day.get('calendar_date'),
                'slot_index': i,
                'name': slot.get('name'),
                'category': slot.get('category'),
                'location': slot.get('location'),
                'cost': slot.get('cost'),
                'link_count': len(links),
                'domains': ', '.join(sorted({urlparse(link).netloc.lower().replace('www.', '') for link in links if isinstance(link, str)})),
                'error_mentions': len(ERROR_RE.findall(text)),
                'text': text,
            })
    for agent, hist in (outputs.get('message_histories') or {}).items():
        messages = hist.get('messages') or []
        history_rows.append({'root_run_id': root_id, 'agent': agent, 'message_count': len(messages), 'chars': sum(len(str(m.get('content', ''))) for m in messages)})

root_summary = pd.DataFrame(summary_rows)
tasks = pd.DataFrame(task_rows)
slots = pd.DataFrame(slot_rows)
todos = pd.DataFrame(todo_rows)
histories = pd.DataFrame(history_rows)

display(root_summary)
if not slots.empty:
    display(slots[['root_run_id', 'day_index', 'slot_index', 'category', 'name', 'cost', 'link_count', 'domains']])

In [ ]:
if not root_summary.empty:
    fig = px.bar(root_summary, x='root_run_id', y=['task_count', 'day_count', 'slot_count', 'link_count'], barmode='group', title='Trace-Level Artifact Volume')
    fig.update_layout(xaxis_title='Root run', yaxis_title='Count', legend_title='Metric')
    fig.show()

if not slots.empty:
    day_costs = slots.groupby(['root_run_id', 'day_index'], as_index=False)['cost'].sum(numeric_only=True)
    fig = px.bar(day_costs, x='day_index', y='cost', color='root_run_id', barmode='group', title='Estimated Cost By Day')
    fig.update_layout(xaxis_title='Day', yaxis_title='Cost')
    fig.show()

    fig = px.histogram(slots, x='category', color='root_run_id', title='Itinerary Slot Categories')
    fig.update_layout(xaxis_title='Slot category', yaxis_title='Count')
    fig.show()

## Repeated Tool Calls

The raw export can contain many observations of a single tool call. For error analysis, use both views:

- raw rows show trace-observation bloat and LangSmith depth duplication
- de-duplicated calls show the actual agent action count

In [ ]:
if tool_raw.empty:
    display(Markdown('No tool call file found.'))
else:
    raw_counts = tool_raw.groupby('tool_name').size().rename('raw_rows')
    unique_counts = tool_calls.groupby('tool_name').size().rename('unique_calls')
    repeated = pd.concat([raw_counts, unique_counts], axis=1).fillna(0).astype(int).reset_index()
    repeated['export_duplication_factor'] = (repeated['raw_rows'] / repeated['unique_calls'].replace(0, np.nan)).round(1)
    repeated = repeated.sort_values(['unique_calls', 'raw_rows'], ascending=False)
    display(repeated)

    fig = px.bar(repeated, x='tool_name', y=['unique_calls', 'raw_rows'], barmode='group', title='Tool Calls: Actual Unique Calls vs Raw Export Observations')
    fig.update_layout(xaxis_title='Tool', yaxis_title='Count')
    fig.show()

    fig = px.bar(repeated.sort_values('export_duplication_factor', ascending=False), x='tool_name', y='export_duplication_factor', title='Raw Export Duplication Factor By Tool')
    fig.update_layout(xaxis_title='Tool', yaxis_title='Raw rows per unique call')
    fig.show()

## Repeated And Near-Duplicate Queries

Exact repeats are based on normalized query text. Near repeats use TF-IDF cosine similarity on de-duplicated search-like tool calls. Repetition is not automatically wrong: `add_slot` and `write_todos` naturally repeat. Risk is higher when search tools repeat the same or nearly identical query without adding date, location, budget, or other constraints.

In [ ]:
if tool_calls.empty:
    exact_repeats = pd.DataFrame()
else:
    exact_repeats = (
        tool_calls[tool_calls['query_norm'].ne('')]
        .groupby(['thread_id', 'tool_name', 'query_norm'], as_index=False)
        .agg(repeat_count=('tool_call_id', 'nunique'), first_seq=('seq', 'min'), last_seq=('seq', 'max'), example_query=('query', 'first'))
        .query('repeat_count > 1')
        .sort_values(['repeat_count', 'tool_name'], ascending=[False, True])
    )
display(exact_repeats)

near_rows = []
search_df = tool_calls[tool_calls['is_search_tool'] & tool_calls['query_norm'].str.len().gt(10)].copy() if not tool_calls.empty else pd.DataFrame()
for (thread_id, tool_name), group in search_df.groupby(['thread_id', 'tool_name']):
    if len(group) < 2:
        continue
    texts = group['query_norm'].tolist()
    tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=1).fit_transform(texts)
    sim = cosine_similarity(tfidf)
    idx = list(group.index)
    for i in range(len(idx)):
        for j in range(i + 1, len(idx)):
            if sim[i, j] >= 0.86:
                near_rows.append({
                    'thread_id': thread_id,
                    'tool_name': tool_name,
                    'similarity': round(float(sim[i, j]), 3),
                    'seq_a': int(group.loc[idx[i], 'seq']),
                    'seq_b': int(group.loc[idx[j], 'seq']),
                    'query_a': group.loc[idx[i], 'query'],
                    'query_b': group.loc[idx[j], 'query'],
                })
near_repeats = pd.DataFrame(near_rows).sort_values('similarity', ascending=False) if near_rows else pd.DataFrame(columns=['thread_id', 'tool_name', 'similarity', 'seq_a', 'seq_b', 'query_a', 'query_b'])
display(near_repeats.head(50))

if not search_df.empty:
    fig = px.histogram(search_df, x='tool_name', color='tool_name', title='Search-Like Tool Calls By Tool')
    fig.update_layout(showlegend=False, xaxis_title='Search tool', yaxis_title='Unique calls')
    fig.show()
if not near_repeats.empty:
    fig = px.scatter(near_repeats, x='seq_a', y='seq_b', color='tool_name', size='similarity', hover_data=['query_a', 'query_b'], title='Near-Duplicate Search Query Pairs')
    fig.update_layout(xaxis_title='Earlier sequence index', yaxis_title='Later sequence index')
    fig.show()

## Dead-Loop Candidates

A dead-loop candidate is a repeated search/query action with little or no argument change. This cell also flags local sequences where the same tool is called many times in a row. These are candidates for manual review, not final labels.

In [ ]:
loop_rows = []
if not tool_calls.empty:
    for thread_id, group in tool_calls.sort_values(['thread_id', 'seq']).groupby('thread_id'):
        group = group.reset_index(drop=True)
        run_tool, start, last_tool = None, 0, None
        for i, row in group.iterrows():
            tool = row['tool_name']
            if tool != last_tool:
                if last_tool is not None and i - start >= 4:
                    segment = group.iloc[start:i]
                    loop_rows.append({
                        'thread_id': thread_id,
                        'loop_type': 'same_tool_burst',
                        'tool_name': last_tool,
                        'start_seq': int(segment['seq'].min()),
                        'end_seq': int(segment['seq'].max()),
                        'length': len(segment),
                        'unique_queries': segment['query_norm'].nunique(),
                        'example_query': segment['query'].iloc[0],
                    })
                start = i
                last_tool = tool
        if last_tool is not None and len(group) - start >= 4:
            segment = group.iloc[start:]
            loop_rows.append({
                'thread_id': thread_id, 'loop_type': 'same_tool_burst', 'tool_name': last_tool,
                'start_seq': int(segment['seq'].min()), 'end_seq': int(segment['seq'].max()),
                'length': len(segment), 'unique_queries': segment['query_norm'].nunique(), 'example_query': segment['query'].iloc[0],
            })

    for _, row in exact_repeats.iterrows():
        if row['tool_name'] in SEARCH_TOOLS:
            loop_rows.append({
                'thread_id': row['thread_id'], 'loop_type': 'exact_search_repeat', 'tool_name': row['tool_name'],
                'start_seq': int(row['first_seq']), 'end_seq': int(row['last_seq']),
                'length': int(row['repeat_count']), 'unique_queries': 1, 'example_query': row['example_query'],
            })

dead_loops = pd.DataFrame(loop_rows).sort_values(['length', 'unique_queries'], ascending=[False, True]) if loop_rows else pd.DataFrame(columns=['thread_id', 'loop_type', 'tool_name', 'start_seq', 'end_seq', 'length', 'unique_queries', 'example_query'])
display(dead_loops)

if not dead_loops.empty:
    fig = px.scatter(dead_loops, x='start_seq', y='length', color='tool_name', symbol='loop_type', hover_data=['example_query'], title='Dead-Loop Candidate Bursts')
    fig.update_layout(xaxis_title='Start sequence', yaxis_title='Burst/repeat length')
    fig.show()

if not tool_calls.empty:
    timeline = tool_calls.sort_values(['thread_id', 'seq']).copy()
    fig = px.scatter(timeline, x='seq', y='tool_name', color='is_search_tool', facet_row='thread_id', hover_data=['query'], title='Tool Timeline By Trace')
    fig.update_layout(xaxis_title='De-duplicated tool sequence', yaxis_title='Tool')
    fig.show()

## Missing Parameter Risk

The paper highlights missing mandatory parameters as a tool-use error class. This heuristic flags search-like calls that mention a domain needing temporal or spatial constraints but do not visibly include a date or location token.

In [ ]:
if tool_calls.empty:
    param_risk = pd.DataFrame()
else:
    param_risk = tool_calls[tool_calls['is_search_tool'] & (tool_calls['missing_date_hint'] | tool_calls['missing_location_hint'])].copy()
    param_risk['risk_reason'] = np.select(
        [param_risk['missing_date_hint'] & param_risk['missing_location_hint'], param_risk['missing_date_hint'], param_risk['missing_location_hint']],
        ['missing visible date and location', 'missing visible date', 'missing visible location'],
        default='parameter risk',
    )
display(param_risk[['thread_id', 'seq', 'tool_name', 'risk_reason', 'query']].head(100) if not param_risk.empty else param_risk)

if not tool_calls.empty:
    risk_counts = param_risk.groupby(['tool_name', 'risk_reason']).size().reset_index(name='count') if not param_risk.empty else pd.DataFrame(columns=['tool_name', 'risk_reason', 'count'])
    if not risk_counts.empty:
        fig = px.bar(risk_counts, x='tool_name', y='count', color='risk_reason', title='Missing Mandatory Parameter Risk By Search Tool')
        fig.update_layout(xaxis_title='Tool', yaxis_title='Flagged calls')
        fig.show()

## Timeouts, Incomplete Runs, And Explicit Errors

This section scans the flat LangSmith run table for explicit `error` values, timeout language, missing timestamps, and unusually long durations within this export.

In [ ]:
if runs.empty:
    display(Markdown('No flat LangSmith run CSV found.'))
else:
    runs['incomplete_time'] = runs['start_time_dt'].isna() | runs['end_time_dt'].isna()
    duration_threshold = runs['duration_s'].quantile(0.95) if runs['duration_s'].notna().any() else np.nan
    runs['slow_p95_flag'] = runs['duration_s'].ge(duration_threshold) if not np.isnan(duration_threshold) else False
    error_runs = runs[runs['has_error'] | runs['timeout_flag'] | runs['incomplete_time'] | runs['slow_p95_flag']].copy()
    display(error_runs[['id', 'root_run_id', 'thread_id', 'parent_run_id', 'name', 'run_type', 'duration_s', 'has_error', 'timeout_flag', 'incomplete_time', 'slow_p95_flag', 'error']].sort_values('duration_s', ascending=False).head(100))

    by_type = runs.groupby('run_type', dropna=False).agg(
        run_count=('id', 'count'),
        mean_duration_s=('duration_s', 'mean'),
        max_duration_s=('duration_s', 'max'),
        error_count=('has_error', 'sum'),
        timeout_count=('timeout_flag', 'sum'),
    ).reset_index().sort_values('run_count', ascending=False)
    display(by_type)

    fig = px.histogram(runs, x='duration_s', nbins=50, color='run_type', title='Run Duration Distribution')
    fig.update_layout(xaxis_title='Duration (seconds)', yaxis_title='Runs')
    fig.show()

    top_slow = runs.sort_values('duration_s', ascending=False).head(30)
    fig = px.bar(top_slow, x='duration_s', y='name', color='run_type', orientation='h', hover_data=['id', 'parent_run_id'], title='Top Slowest LangSmith Runs')
    fig.update_layout(xaxis_title='Duration (seconds)', yaxis_title='Run name')
    fig.show()

## Cascading Error Analysis

Cascades are hard to prove from traces alone. This notebook treats them as evidence chains where an upstream risky event is followed by downstream plan mutation or validation activity in the same trace. It combines three signals:

- explicit run errors or timeout-like runs
- risky search/tool calls such as missing parameters or redundant search loops
- downstream mutation/review activity after the risky event

The graph below is a candidate map for manual case-study review.

In [ ]:
cascade_events = []

if not tool_calls.empty:
    risky_ids = set(param_risk['tool_call_id']) if 'param_risk' in globals() and not param_risk.empty else set()
    loop_intervals = dead_loops.to_dict('records') if 'dead_loops' in globals() and not dead_loops.empty else []
    for row in tool_calls.itertuples(index=False):
        risk = []
        if row.tool_call_id in risky_ids:
            risk.append('missing_parameter_risk')
        for loop in loop_intervals:
            if loop['thread_id'] == row.thread_id and loop['start_seq'] <= row.seq <= loop['end_seq']:
                risk.append(loop['loop_type'])
        if risk:
            later = tool_calls[(tool_calls['thread_id'].eq(row.thread_id)) & (tool_calls['seq'].gt(row.seq))]
            later_mutations = later[later['is_plan_mutation']]
            cascade_events.append({
                'thread_id': row.thread_id,
                'source_seq': int(row.seq),
                'source_tool': row.tool_name,
                'risk_signal': ', '.join(sorted(set(risk))),
                'source_query': row.query,
                'downstream_tool_calls': len(later),
                'downstream_plan_mutations': len(later_mutations),
                'next_mutation_seq': int(later_mutations['seq'].min()) if not later_mutations.empty else np.nan,
            })

cascades = pd.DataFrame(cascade_events).sort_values(['downstream_plan_mutations', 'downstream_tool_calls'], ascending=False) if cascade_events else pd.DataFrame(columns=['thread_id', 'source_seq', 'source_tool', 'risk_signal', 'source_query', 'downstream_tool_calls', 'downstream_plan_mutations', 'next_mutation_seq'])
display(cascades.head(100))

if not cascades.empty:
    fig = px.scatter(cascades, x='source_seq', y='downstream_plan_mutations', color='source_tool', size='downstream_tool_calls', hover_data=['risk_signal', 'source_query'], title='Cascade Candidates: Risky Tool Events Followed By Plan Mutations')
    fig.update_layout(xaxis_title='Risk source sequence', yaxis_title='Later plan mutation calls')
    fig.show()

if not tool_calls.empty:
    # Compact directed graph: tool-to-tool transitions weighted by sequence adjacency.
    edges = []
    for thread_id, group in tool_calls.sort_values(['thread_id', 'seq']).groupby('thread_id'):
        names = group['tool_name'].tolist()
        for a, b in zip(names, names[1:]):
            edges.append((a, b))
    edge_counts = Counter(edges)
    G = nx.DiGraph()
    for (a, b), weight in edge_counts.items():
        G.add_edge(a, b, weight=weight)
    if len(G):
        pos = nx.spring_layout(G, seed=7, k=0.8)
        edge_x, edge_y = [], []
        for a, b in G.edges():
            x0, y0 = pos[a]
            x1, y1 = pos[b]
            edge_x += [x0, x1, None]
            edge_y += [y0, y1, None]
        edge_trace = go.Scatter(x=edge_x, y=edge_y, line=dict(width=1, color='#999'), hoverinfo='none', mode='lines')
        node_trace = go.Scatter(
            x=[pos[n][0] for n in G.nodes()], y=[pos[n][1] for n in G.nodes()], mode='markers+text', text=list(G.nodes()), textposition='top center',
            marker=dict(size=[12 + 2 * G.degree(n) for n in G.nodes()], color=[G.degree(n) for n in G.nodes()], colorscale='Viridis', showscale=True),
            hovertext=[f'{n}<br>degree={G.degree(n)}' for n in G.nodes()], hoverinfo='text'
        )
        fig = go.Figure(data=[edge_trace, node_trace])
        fig.update_layout(title='Tool Transition Graph', showlegend=False, xaxis=dict(visible=False), yaxis=dict(visible=False), height=650)
        fig.show()

## Evidence And Link-Rot Proxies

The paper notes that TravelPlanner often uses exact links, which improves traceability but increases vulnerability to empty crawls and JavaScript-heavy pages. This section summarizes evidence domains and slots with no links.

In [ ]:
if slots.empty:
    display(Markdown('No TravelPlan slots found in root artifacts.'))
else:
    no_link_slots = slots[slots['link_count'].eq(0)].copy()
    display(no_link_slots[['root_run_id', 'day_index', 'slot_index', 'category', 'name', 'location', 'cost']])

    domain_rows = []
    for _, row in slots.iterrows():
        for domain in [d.strip() for d in str(row['domains']).split(',') if d.strip()]:
            domain_rows.append({'root_run_id': row['root_run_id'], 'category': row['category'], 'domain': domain})
    domains = pd.DataFrame(domain_rows)
    if not domains.empty:
        domain_counts = domains.groupby('domain').size().reset_index(name='count').sort_values('count', ascending=False)
        display(domain_counts.head(30))
        fig = px.bar(domain_counts.head(25), x='count', y='domain', orientation='h', title='Top Evidence Link Domains')
        fig.update_layout(xaxis_title='Slots linking to domain', yaxis_title='Domain')
        fig.show()

    fig = px.bar(slots.groupby(['category'], as_index=False)['link_count'].mean(), x='category', y='link_count', title='Average Evidence Links Per Slot Category')
    fig.update_layout(xaxis_title='Category', yaxis_title='Average links')
    fig.show()

## Final Trace Risk Scorecard

This scorecard combines the notebook heuristics into a trace-level table for project evaluation. Treat scores as triage signals for qualitative review, not as ground-truth labels.

In [ ]:
score = manifest[['root_run_id', 'thread_id', 'query', 'validation_passed', 'validation_attempts', 'days']].copy()

if not tool_calls.empty:
    by_thread = tool_calls.groupby('thread_id').agg(unique_tool_calls=('tool_call_id', 'nunique'), unique_tools=('tool_name', 'nunique'), search_calls=('is_search_tool', 'sum'), plan_mutations=('is_plan_mutation', 'sum')).reset_index()
    score = score.merge(by_thread, on='thread_id', how='left')
else:
    score[['unique_tool_calls', 'unique_tools', 'search_calls', 'plan_mutations']] = 0

if not exact_repeats.empty:
    repeat_counts = exact_repeats.groupby('thread_id')['repeat_count'].sum().reset_index(name='exact_repeat_calls')
    score = score.merge(repeat_counts, on='thread_id', how='left')
else:
    score['exact_repeat_calls'] = 0

score['near_duplicate_pairs'] = 0
if 'near_repeats' in globals() and not near_repeats.empty:
    nd = near_repeats.groupby('thread_id').size().reset_index(name='near_duplicate_pairs')
    score = score.drop(columns=['near_duplicate_pairs']).merge(nd, on='thread_id', how='left')

score['dead_loop_candidates'] = 0
if 'dead_loops' in globals() and not dead_loops.empty:
    dl = dead_loops.groupby('thread_id').size().reset_index(name='dead_loop_candidates')
    score = score.drop(columns=['dead_loop_candidates']).merge(dl, on='thread_id', how='left')

score['missing_param_risk_calls'] = 0
if 'param_risk' in globals() and not param_risk.empty:
    pr = param_risk.groupby('thread_id').size().reset_index(name='missing_param_risk_calls')
    score = score.drop(columns=['missing_param_risk_calls']).merge(pr, on='thread_id', how='left')

score['cascade_candidates'] = 0
if 'cascades' in globals() and not cascades.empty:
    cc = cascades.groupby('thread_id').size().reset_index(name='cascade_candidates')
    score = score.drop(columns=['cascade_candidates']).merge(cc, on='thread_id', how='left')

if not runs.empty:
    run_risk = runs.groupby('thread_id').agg(run_errors=('has_error', 'sum'), timeout_flags=('timeout_flag', 'sum'), slow_runs=('slow_p95_flag', 'sum')).reset_index()
    score = score.merge(run_risk, on='thread_id', how='left')
else:
    score[['run_errors', 'timeout_flags', 'slow_runs']] = 0

score = score.fillna(0)
risk_cols = ['exact_repeat_calls', 'near_duplicate_pairs', 'dead_loop_candidates', 'missing_param_risk_calls', 'cascade_candidates', 'run_errors', 'timeout_flags', 'slow_runs']
score['risk_score'] = score[risk_cols].sum(axis=1) + (~score['validation_passed'].astype(bool)).astype(int) * 5 + score['validation_attempts'].clip(lower=1).sub(1) * 2
score['risk_level'] = pd.cut(score['risk_score'], bins=[-1, 0, 5, 20, np.inf], labels=['clean', 'low', 'medium', 'high'])
display(score.sort_values('risk_score', ascending=False))

fig = px.bar(score.sort_values('risk_score', ascending=False), x='root_run_id', y=risk_cols, title='Trace Risk Score Components', barmode='stack')
fig.update_layout(xaxis_title='Root run', yaxis_title='Heuristic count')
fig.show()

display(Markdown('### Interpretation Template'))
for row in score.sort_values('risk_score', ascending=False).itertuples(index=False):
    display(Markdown(f'- **{row.root_run_id}**: risk_level={row.risk_level}, risk_score={row.risk_score}. validation_passed={row.validation_passed}, attempts={row.validation_attempts}, unique_tool_calls={row.unique_tool_calls}, dead_loop_candidates={row.dead_loop_candidates}, cascade_candidates={row.cascade_candidates}.'))